# 03 · 抽样分布与中心极限定理 Sampling Distribution & CLT

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**抽样分布**（sampling distribution）的概念：统计量在重复采样下的分布。
2. 陈述并应用**中心极限定理**（Central Limit Theorem, CLT）。
3. 计算**标准误**（Standard Error, SE = s/√n）并理解其含义。
4. 用 `sample_means`/`standard_error` 函数模拟 CLT。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 抽样分布 Sampling Distribution

对总体重复抽取大小为 $n$ 的样本，每次计算统计量（如样本均值 $\bar{X}$），这些统计量的分布称为**抽样分布**。

关键性质（总体均值 $\mu$，方差 $\sigma^2$）：
$$E[\bar{X}] = \mu, \quad \text{Var}(\bar{X}) = \frac{\sigma^2}{n}$$

---

### 中心极限定理 CLT

**定理**：设 $X_1, \ldots, X_n$ 独立同分布，$E[X_i]=\mu$，$\text{Var}(X_i)=\sigma^2 < \infty$，则：

$$\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \xrightarrow{d} \mathcal{N}(0, 1), \quad n \to \infty$$

**直觉**：无论原始数据是什么分布，样本均值在大样本下趋近正态分布。
实际中 $n \geq 30$ 通常足够（对称分布可更小）。

---

### 标准误 Standard Error

**标准误**（SE）度量样本均值的不确定性：

$$SE = \frac{s}{\sqrt{n}}$$

其中 $s = \sqrt{\frac{1}{n-1}\sum(x_i-\bar{x})^2}$ 是样本标准差（ddof=1）。

- $SE$ 越小 → 估计越精确
- 增大样本量 $n$ 可以减小 $SE$（但需要 4 倍样本才能将 SE 减半）


## 示例 Worked Example

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
from scipy import stats


def standard_error(x) -> float:
    """样本均值的标准误：s/√n（s 用 ddof=1）。"""
    x = np.asarray(x, dtype=float)
    return float(x.std(ddof=1) / np.sqrt(len(x)))


def sample_means(data, n: int, reps: int, seed: int = 0):
    """有放回抽样 reps 次，每次 n 个，返回每次的样本均值（长度 reps 的数组）。"""
    rng = np.random.default_rng(seed)
    data = np.asarray(data, dtype=float)
    return np.array([rng.choice(data, size=n, replace=True).mean() for _ in range(reps)])


# 总体：指数分布（明显非正态）
rng = np.random.default_rng(42)
population = rng.exponential(scale=2.0, size=100_000)

print(f"总体形状: 指数分布 (均值=2.0)")
print(f"总体均值={population.mean():.4f}, 总体标准差={population.std():.4f}")

# 演示 CLT：不同样本量的抽样分布
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle("CLT 演示：指数总体的样本均值分布（reps=3000）", fontsize=11)

for ax, n in zip(axes, [1, 5, 30, 100]):
    means = sample_means(population, n=n, reps=3000, seed=n)
    ax.hist(means, bins=40, density=True, color="steelblue", edgecolor="white", alpha=0.8)
    # 叠加理论正态曲线
    se_theory = population.std() / np.sqrt(n)
    x = np.linspace(means.min(), means.max(), 200)
    ax.plot(x, stats.norm.pdf(x, population.mean(), se_theory), "r-", lw=2, label="CLT")
    ax.set_title(f"n={n}")
    ax.set_xlabel("样本均值")
    ax.legend(fontsize=8)

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "clt_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("\n图像已保存到:", _outpath)


In [ ]:
import numpy as np

# 标准误演示：样本量增大，SE 减小
rng = np.random.default_rng(100)
population = rng.normal(5.0, 2.0, 100_000)

print("=== 标准误 SE = s/√n ===")
print(f"{'n':>6}  {'样本均值':>10}  {'SE':>8}  {'CLT-SE(σ/√n)':>12}")
print("-" * 50)
for n in [10, 30, 100, 500, 2000]:
    x = rng.choice(population, size=n, replace=False)
    se = x.std(ddof=1) / np.sqrt(n)
    clt_se = 2.0 / np.sqrt(n)   # 真实σ=2.0
    print(f"{n:>6}  {x.mean():>10.4f}  {se:>8.4f}  {clt_se:>12.4f}")


## 动手 Exercise

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# 练习：验证 CLT 对均匀分布的适用性
# Uniform(0,1): E[X]=0.5, Var(X)=1/12

def sample_means(data, n: int, reps: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    data = np.asarray(data, dtype=float)
    return np.array([rng.choice(data, size=n, replace=True).mean() for _ in range(reps)])

def standard_error(x) -> float:
    x = np.asarray(x, dtype=float)
    return float(x.std(ddof=1) / np.sqrt(len(x)))

rng = np.random.default_rng(55)
population = rng.uniform(0, 1, 50_000)

# 取 n=50 的 2000 次抽样均值
n, reps = 50, 2000
means = sample_means(population, n=n, reps=reps, seed=0)

print(f"总体: Uniform(0,1), 真实 μ=0.5, 真实 SE=√(1/12/{n})={1/(12*n)**0.5:.4f}")
print(f"样本均值的均值: {means.mean():.4f}")
print(f"样本均值的标准差: {means.std():.4f}  (≈ SE)")

# 取一个样本计算 SE
one_sample = rng.choice(population, size=n, replace=True)
print(f"\n单样本(n={n}) SE = {standard_error(one_sample):.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(means, bins=30, density=True, color="darkorange", edgecolor="white", alpha=0.8)
ax.set_title(f"CLT: Uniform(0,1) 样本均值分布 (n={n}, reps={reps})")
ax.set_xlabel("样本均值")
ax.set_ylabel("密度")
fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "clt_uniform_exercise.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)


## 延伸阅读 Further Reading

1. **《All of Statistics》— Larry Wasserman**：第 5 章（收敛概念与 CLT）
2. **StatQuest: Central Limit Theorem**（YouTube）
3. **《Probability and Statistics for Machine Learning》**：第 7 章
4. **SciPy stats 文档**：<https://docs.scipy.org/doc/scipy/reference/stats.html>


## 关联练习 Related Assignments

本课对应练习包：**`w4-clt`**（实现 `sample_means` 与 `standard_error`）

```bash
python check.py 02-clt
```

> 能力标签：**SH6** · threshold ≥ 0.7
